# Generating embeddings

## Pre-processing

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('../data/petitions.csv')

data.head()

,id,public_petition_text,reason_category
0,3168490,снег на дороге,Благоустройство
1,3219678,очистить кабельный киоск от рекламы,Благоустройство
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД
4,3336285,Граффити,Благоустройство


In [3]:
petitions = data['public_petition_text'].dropna().astype(str)

petitions.head()

0                                       снег на дороге
1                  очистить кабельный киоск от рекламы
2    Просим убрать все деревья и кустарники, которы...
3    Неудовлетворительное состояние парадной - надп...
4                                             Граффити
Name: public_petition_text, dtype: str

In [4]:
import re
import nltk
import pymorphy3
from razdel import tokenize
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):

    text = text.lower()

    # Удаление URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Удаление email
    text = re.sub(r'\S+@\S+', '', text)

    # Замена числительных на специальный токен
    text = re.sub(r'\d+', 'NUM', text)

    # Токенизация через razdel
    tokens = [token.text for token in tokenize(text)]

    # Убирает точки, запятые, скобки, тире, кавычки и смайлики
    clean_tokens = [
        t for t in tokens 
        if re.search(r'[а-яёa-z]', t) or t == 'NUM'
    ]

    # Лемматизация
    lemmatized_words = [morph.parse(word)[0].normal_form for word in clean_tokens]

    # Удаление стоп-слов
    filtered_tokens = [word for word in lemmatized_words if word not in russian_stopwords]

    return filtered_tokens

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/irinaaristova/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Dataset tokenization

In [5]:
from tqdm.auto import tqdm

tqdm.pandas()

processed_tokens = petitions.progress_apply(preprocess_text)

processed_tokens = processed_tokens[processed_tokens.apply(len) > 0].reset_index(drop=True)

processed_tokens.to_pickle('../data/petitions_tokens.pkl')

  0%|          | 0/59889 [00:00<?, ?it/s]

In [6]:
processed_tokens = pd.read_pickle('../data/petitions_tokens.pkl')

### CBoW and SkipGram

In [7]:
from sklearn.decomposition import PCA
import numpy as np

word_counts = {}
for text in processed_tokens:
    for word in text:
        word_counts[word] = word_counts.get(word, 0) + 1

word_to_idx = {'<UNK>': 0}

for word, count in word_counts.items():
    if count >= 5:
        word_to_idx[word] = len(word_to_idx)

idx_to_word = {idx: word for word, idx in word_to_idx.items()}

VOCAB_SIZE = len(word_to_idx)

In [8]:
import torch
from torch import nn

context_window = 2 
cbow_x, cbow_y = [], []
sg_x, sg_y = [], []

for text in processed_tokens:
    indices = [word_to_idx.get(w, 0) for w in text]
    if len(indices) < context_window * 2 + 1:
        continue
        
    for i in range(context_window, len(indices) - context_window):
        center = indices[i]
        context = indices[i - context_window : i] + indices[i + 1 : i + context_window + 1]
        
        cbow_x.append(context)
        cbow_y.append(center)
        
        for ctx_word in context:
            sg_x.append(center)
            sg_y.append(ctx_word)

In [9]:
cbow_x = torch.tensor(cbow_x, dtype=torch.long)
cbow_y = torch.tensor(cbow_y, dtype=torch.long)
sg_x = torch.tensor(sg_x, dtype=torch.long)
sg_y = torch.tensor(sg_y, dtype=torch.long)

In [10]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        return self.linear(self.embeddings(x).mean(dim=1))

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        return self.linear(self.embeddings(x))

In [11]:
import math

def train_model(model, x_tensor, y_tensor, epochs=10, batch_size=256, lr=0.005):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    dataset_size = x_tensor.size(0)
    
    model.train()
    for epoch in range(epochs):
        indices = torch.randperm(dataset_size)
        total_loss = 0
        
        for i in range(0, dataset_size, batch_size):
            batch_idx = indices[i : i + batch_size]
            batch_x, batch_y = x_tensor[batch_idx], y_tensor[batch_idx]
            
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / (dataset_size // batch_size + 1)
        
        perplexity = math.exp(avg_loss)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Perplexity: {perplexity:.2f}")
        
    return model

EMBED_DIM = 100
cbow_model = train_model(CBOW(VOCAB_SIZE, EMBED_DIM), cbow_x, cbow_y)
sg_model = train_model(SkipGram(VOCAB_SIZE, EMBED_DIM), sg_x, sg_y)

Epoch 1/10 | Loss: 5.3629 | Perplexity: 213.34
Epoch 2/10 | Loss: 4.3182 | Perplexity: 75.05
Epoch 3/10 | Loss: 3.9475 | Perplexity: 51.80
Epoch 4/10 | Loss: 3.7174 | Perplexity: 41.16
Epoch 5/10 | Loss: 3.5523 | Perplexity: 34.89
Epoch 6/10 | Loss: 3.4221 | Perplexity: 30.64
Epoch 7/10 | Loss: 3.3200 | Perplexity: 27.66
Epoch 8/10 | Loss: 3.2366 | Perplexity: 25.45
Epoch 9/10 | Loss: 3.1670 | Perplexity: 23.74
Epoch 10/10 | Loss: 3.1084 | Perplexity: 22.39
Epoch 1/10 | Loss: 6.2272 | Perplexity: 506.33
Epoch 2/10 | Loss: 5.8528 | Perplexity: 348.22
Epoch 3/10 | Loss: 5.7527 | Perplexity: 315.04
Epoch 4/10 | Loss: 5.7004 | Perplexity: 298.98
Epoch 5/10 | Loss: 5.6692 | Perplexity: 289.81
Epoch 6/10 | Loss: 5.6502 | Perplexity: 284.35
Epoch 7/10 | Loss: 5.6374 | Perplexity: 280.72
Epoch 8/10 | Loss: 5.6275 | Perplexity: 277.98
Epoch 9/10 | Loss: 5.6217 | Perplexity: 276.35
Epoch 10/10 | Loss: 5.6163 | Perplexity: 274.86


In [ ]:
def extract_and_save_embeddings(model, prefix_name, target_dim=3):
    embeddings = model.linear.weight.detach().numpy() ### linear or embeddings
    
    pca = PCA(n_components=target_dim)
    embeddings_pca = pca.fit_transform(embeddings)
    
    pd.DataFrame(embeddings_pca).to_csv(
        f'{prefix_name}_vectors.tsv', sep='\t', index=False, header=False
    )
    
    words = [idx_to_word[i] for i in range(VOCAB_SIZE)]
    pd.DataFrame(words).to_csv(
        f'{prefix_name}_metadata.tsv', sep='\t', index=False, header=False
    )

extract_and_save_embeddings(cbow_model, 'cbow')    
extract_and_save_embeddings(sg_model, 'skipgram')

In [18]:
words = [idx_to_word[i] for i in range(VOCAB_SIZE)]

with open('vocab_words.txt', 'w', encoding='utf-8') as f:
    for word in words:
        f.write(f"{word}\n")